# Turkish-Gemma QLoRA ince ayarı

Bu defter kısaltma genişletme modelini eğitir ve doğrulama kümesinde sıfır atış modelle karşılaştırır.

1. **Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU** seçin.
2. Hücreleri sırayla çalıştırın. Adaptör ve sonuçlar Google Drive'a yazılır; Colab bağlantısı koparsa eğitim hücresini yeniden çalıştırmanız yeterli, son epoch'tan devam eder.

Ayarlar teknik plandan gelir (r=16, alpha=32, lr=2e-4, 3 epoch, batch 4x4). Ayrıntı: `docs/setup.md`, "İnce ayar" bölümü.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/gaze-aac"
ADAPTER = f"{DRIVE}/turkish-gemma-qlora"
!mkdir -p "$DRIVE/sonuclar"

Kurulum birkaç dakika sürer. Unsloth eğitimi yaklaşık iki kat hızlandırır; kurulumu başarısız olursa betik PEFT ile devam eder.

In [ ]:
%cd /content
!test -d FINAL-YEAR-PROJECT || git clone -q https://github.com/tahaaoztrkk/FINAL-YEAR-PROJECT.git
!git -C FINAL-YEAR-PROJECT pull -q
%cd /content/FINAL-YEAR-PROJECT/backend
!git log --oneline -1
!pip install -q -e ".[llm]"
!pip install -q unsloth

Model indirilirken 401 ya da 403 hatası çıkarsa Hugging Face hesabınızla giriş yapın (model sayfasında lisans onayı istenebilir). Hata yoksa bu hücreyi atlayın.

In [ ]:
# from huggingface_hub import notebook_login
# notebook_login()

Derlemi bölüp eğitim çiftlerini üretir, sonra modeli yüklemeden veriyi denetler: kayıt sayısı, sızıntı, adım sayısı.

In [ ]:
!python -m gaze_aac.data.build_corpus
!python -m gaze_aac.nlp.finetune --dry-run

Eğitim. Her epoch sonunda doğrulama kaybı ölçülür ve en iyi epoch adaptör olarak kaydedilir. Bitince klasörde `training_info.json` olur; rapordaki ayar tablosu oradan alınır.

In [ ]:
!python -m gaze_aac.nlp.finetune --out "$ADAPTER" --resume
!cat "$ADAPTER/training_info.json"

## Karşılaştırma

Aynı doğrulama cümleleri önce ince ayarsız modelle (sıfır atış taban çizgisi), sonra adaptörle ölçülür. Her koşu modeli bir kez yükler; hızlı bir deneme için komutlara `--limit 5` ekleyin.

In [ ]:
!python -m gaze_aac.evaluation.simulate --providers ngram hf_local --out "$DRIVE/sonuclar/val_sifir_atis.csv"

In [ ]:
!python -m gaze_aac.evaluation.simulate --providers hf_local --adapter "$ADAPTER" --out "$DRIVE/sonuclar/val_lora.csv"

In [ ]:
import pandas as pd

table = pd.concat(
    [pd.read_csv(f"{DRIVE}/sonuclar/val_sifir_atis.csv"), pd.read_csv(f"{DRIVE}/sonuclar/val_lora.csv")]
)
table.pivot_table(index="sema", columns="saglayici", values="ksr")